# Colab: SST preprocessing to GPT-2 Small sentiment-direction results

This notebook runs the repository's end-to-end reproduction workflow on a Colab GPU:

1. clone and install the project plus the reference SST source;
2. securely read `HF_TOKEN` from Colab Secrets;
3. preprocess Stanford Sentiment Treebank (SST) with Pythia-1.4B and optionally publish all generated dataset configurations to the Hugging Face Hub;
4. fit GPT-2 Small mean-difference, K-means, logistic-regression, PCA, DAS 1D/2D/3D, and random-control directions;
5. perform all-position directional patching on held-out ToyMovieReview and Pythia-filtered SST;
6. generate, inspect, plot, and download the saved results.

> The project's reproduction runner calls the package implementation directly. The notebook contains no separate fitting, patching, or metric logic.


## Before running

1. In Colab, choose **Runtime → Change runtime type → T4 GPU** (or a larger GPU).
2. Open the **Secrets** panel (key icon), add a secret named `HF_TOKEN`, and enable notebook access. Use a Hugging Face user access token with **write** permission if `PUSH_SST_TO_HUB = True`.
3. Do not paste an `hf_...` token into any code cell. This notebook supplies the secret only to the publishing subprocess and never prints or stores it in notebook output.
4. The smoke run is a pipeline check. It is not a reportable reproduction. The full run scans all 13 GPT-2 Small residual boundaries and is substantially more expensive, especially for DAS.


In [ ]:
# User settings
USE_GOOGLE_DRIVE = True  # Persist fitted direction checkpoints across Colab sessions.

RUN_SST_PREPROCESSING = True
PUSH_SST_TO_HUB = True
HF_DATASET_REPO_ID = "kokolamba/sentiment-manifold-sst-pythia-1.4b"  # Blank => <your-HF-user>/sentiment-manifold-sst-pythia-1.4b
PUBLISH_PRIVATE = True
SST_BATCH_SIZE = 32  # Reduce to 8 or 4 if Pythia preprocessing runs out of GPU memory.

RUN_SMOKE = True
RUN_FULL_REPRODUCTION = False  # Set True only after the smoke run succeeds.
FULL_METHODS = [
    "mean_diff",
    "kmeans",
    "logistic_regression",
    "pca",
    "das",
    "das2d",
    "das3d",
    "random",
]
DEVICE = "cuda"


## 1. Clone and install the two repositories

The SST files are read from the authors' reference repository. Its revision is pinned below so the input files do not silently change. If a checkout already exists in the Colab runtime, it is reused.


In [ ]:
from pathlib import Path
import os
import shlex
import subprocess
import sys

COLAB_ROOT = Path("/content")
PROJECT_ROOT = COLAB_ROOT / "sentiment-manifold"
REFERENCE_ROOT = COLAB_ROOT / "eliciting-latent-sentiment"
PROJECT_URL = "https://github.com/Adefioye/sentiment-manifold.git"
REFERENCE_URL = "https://github.com/curt-tigges/eliciting-latent-sentiment.git"
REFERENCE_REVISION = "037cbc9c18867e03da14ceace612ab7e0f9449f7"

def checked_run(command, *, cwd=None, env=None):
    command = [str(part) for part in command]
    print("+", shlex.join(command))
    return subprocess.run(command, cwd=cwd, env=env, check=True)

if not (PROJECT_ROOT / ".git").is_dir():
    checked_run(["git", "clone", PROJECT_URL, PROJECT_ROOT])
else:
    print(f"Reusing {PROJECT_ROOT}")

reference_was_cloned = not (REFERENCE_ROOT / ".git").is_dir()
if reference_was_cloned:
    checked_run(["git", "clone", REFERENCE_URL, REFERENCE_ROOT])
else:
    print(f"Reusing {REFERENCE_ROOT}")
if reference_was_cloned:
    checked_run(["git", "checkout", "--detach", REFERENCE_REVISION], cwd=REFERENCE_ROOT)
else:
    existing_reference_revision = subprocess.check_output(
        ["git", "rev-parse", "HEAD"], cwd=REFERENCE_ROOT, text=True
    ).strip()
    if existing_reference_revision != REFERENCE_REVISION:
        raise RuntimeError(
            f"Existing reference checkout is {existing_reference_revision}, expected "
            f"{REFERENCE_REVISION}. Use a fresh Colab runtime rather than overwriting it."
        )

checked_run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{PROJECT_ROOT}[notebooks]"])
os.chdir(PROJECT_ROOT)
print("Project revision:", subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=PROJECT_ROOT, text=True).strip())
print("Reference revision:", subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REFERENCE_ROOT, text=True).strip())


## 2. Verify the accelerator and choose storage locations

Lightweight CSV/JSON/PNG results stay on the Colab instance. Only reusable fitted directions are placed on Google Drive when enabled. The runner is resumable and uses configuration fingerprints to prevent incompatible checkpoints from being reused.


In [ ]:
import torch
from sentiment_manifold.storage import maybe_mount_google_drive

if DEVICE == "cuda" and not torch.cuda.is_available():
    raise RuntimeError("CUDA is unavailable. Enable a GPU runtime in Colab before continuing.")

maybe_mount_google_drive(USE_GOOGLE_DRIVE)
CHECKPOINT_ROOT = (
    Path("/content/drive/MyDrive/sentiment-manifold/checkpoints")
    if USE_GOOGLE_DRIVE
    else PROJECT_ROOT / "checkpoints"
)
RESULT_ROOT = PROJECT_ROOT / "outputs" / "colab"
CHECKPOINT_ROOT.mkdir(parents=True, exist_ok=True)
RESULT_ROOT.mkdir(parents=True, exist_ok=True)

print("PyTorch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
print("Results:", RESULT_ROOT)
print("Direction checkpoints:", CHECKPOINT_ROOT)


## 3. Securely validate the Hugging Face secret

The token is retrieved only at runtime from Colab Secrets. It is not assigned to a persistent environment variable, included in a command argument, or printed.


In [ ]:
def get_colab_secret(name):
    try:
        from google.colab import userdata
    except ImportError as exc:
        raise RuntimeError("This secret helper is intended for Google Colab.") from exc
    try:
        value = userdata.get(name)
    except Exception as exc:
        raise RuntimeError(
            f"Add {name!r} in Colab's Secrets panel and enable notebook access."
        ) from exc
    if not value:
        raise RuntimeError(f"Colab secret {name!r} is empty or unavailable.")
    return value

if PUSH_SST_TO_HUB:
    from huggingface_hub import HfApi

    _token = get_colab_secret("HF_TOKEN")
    _account = HfApi(token=_token).whoami()["name"]
    del _token
    print(f"Authenticated to Hugging Face as {_account}. Token value was not displayed.")
else:
    print("Hub publishing is disabled; no Hugging Face write token is required.")


## 4. Preprocess SST with Pythia-1.4B and publish it

This creates both repository-supported binarizations and ten Hugging Face dataset configurations. The reproduction itself consumes `tigges_pythia_correct`, then reconstructs equal-GPT-2-token-length pairs locally.

Publishing defaults to a private dataset. The literal token is added only to a temporary child-process environment and removed immediately after the command returns. If preprocessing output already exists, this cell skips it to avoid overwriting a completed dataset. Use a fresh Colab runtime if you intentionally need to rebuild it.


In [ ]:
SST_OUTPUT_DIR = PROJECT_ROOT / "data/processed/sst-pythia-1.4b"
SST_METADATA = SST_OUTPUT_DIR / "metadata.json"

if RUN_SST_PREPROCESSING and not SST_METADATA.exists():
    command = [
        sys.executable,
        "-m",
        "sentiment_manifold",
        "preprocess-sst",
        "--sst-root",
        REFERENCE_ROOT / "stanfordSentimentTreebank",
        "--output-dir",
        SST_OUTPUT_DIR,
        "--device",
        DEVICE,
        "--dtype",
        "auto",
        "--batch-size",
        str(SST_BATCH_SIZE),
        "--binarization",
        "both",
    ]
    child_env = None
    if PUSH_SST_TO_HUB:
        command.append("--push-to-hub")
        command.append("--private" if PUBLISH_PRIVATE else "--public")
        if HF_DATASET_REPO_ID.strip():
            command.extend(["--hub-repo-id", HF_DATASET_REPO_ID.strip()])
        child_env = os.environ.copy()
        child_env["HF_TOKEN"] = get_colab_secret("HF_TOKEN")
    try:
        checked_run(command, cwd=PROJECT_ROOT, env=child_env)
    finally:
        if child_env is not None:
            child_env.pop("HF_TOKEN", None)
            del child_env
elif SST_METADATA.exists():
    print(f"Reusing completed SST preprocessing at {SST_OUTPUT_DIR}")
else:
    raise FileNotFoundError(
        "Processed SST data is missing. Set RUN_SST_PREPROCESSING = True and run this cell."
    )


In [ ]:
import json

sst_metadata = json.loads(SST_METADATA.read_text())
print("Published repository:", sst_metadata.get("hub_repo_id", "not published in this run"))
print(json.dumps(sst_metadata["counts"], indent=2, sort_keys=True))
assert "tigges_pythia_correct" in sst_metadata["dataset_configs"]


## 5. Inspect GPT-2 Small tokenization

This downloads the pinned GPT-2 Small revision and records how much of ToyMovieReview remains single-token under its tokenizer.


In [ ]:
checked_run(
    [
        sys.executable,
        "-m",
        "sentiment_manifold",
        "inspect-data",
        "--config",
        PROJECT_ROOT / "configs/reproduction.yaml",
        "--model",
        "gpt2-small",
        "--device",
        DEVICE,
    ],
    cwd=PROJECT_ROOT,
)


## 6. Run a cheap pipeline smoke test

The smoke configuration uses one residual boundary, two SST pairs, one DAS epoch, and five methods. It still exercises direction fitting, ToyMovieReview/SST pairing, patching, metrics, artifact saving, and plotting. Do not report smoke metrics as research results.


In [ ]:
SMOKE_OUTPUT_ROOT = RESULT_ROOT / "smoke"
SMOKE_RUN_DIR = SMOKE_OUTPUT_ROOT / "gpt2-small"

if RUN_SMOKE:
    checked_run(
        [
            sys.executable,
            "-m",
            "sentiment_manifold",
            "reproduce",
            "--config",
            PROJECT_ROOT / "configs/smoke.yaml",
            "--model",
            "gpt2-small",
            "--device",
            DEVICE,
            "--output-dir",
            SMOKE_OUTPUT_ROOT,
            "--checkpoint-dir",
            CHECKPOINT_ROOT,
        ],
        cwd=PROJECT_ROOT,
    )
    checked_run(
        [sys.executable, "-m", "sentiment_manifold", "plot", "--run-dir", SMOKE_RUN_DIR],
        cwd=PROJECT_ROOT,
    )
else:
    print("Smoke run disabled.")


## 7. Run the full GPT-2 Small reproduction

Set `RUN_FULL_REPRODUCTION = True` in the settings cell, then run this cell. The command scans residual boundaries 0–12 and runs every method listed in `FULL_METHODS`. For each method/layer, the package fits and saves a direction, performs all-position causal patching, and evaluates held-out ToyMovieReview and the `tigges_pythia_correct` SST benchmark.

The run writes progress incrementally and has `resume: true`, so rerunning after a Colab interruption reuses compatible fitted directions. Google Drive persistence is recommended for a long DAS sweep. OpenWebText resample ablation remains off because it is an optional exploratory diagnostic, not part of the requested core benchmark.


In [ ]:
FULL_OUTPUT_ROOT = RESULT_ROOT / "full"
FULL_RUN_DIR = FULL_OUTPUT_ROOT / "gpt2-small"

if RUN_FULL_REPRODUCTION:
    command = [
        sys.executable,
        "-m",
        "sentiment_manifold",
        "reproduce",
        "--config",
        PROJECT_ROOT / "configs/reproduction.yaml",
        "--model",
        "gpt2-small",
        "--device",
        DEVICE,
        "--output-dir",
        FULL_OUTPUT_ROOT,
        "--checkpoint-dir",
        CHECKPOINT_ROOT,
    ]
    for method in FULL_METHODS:
        command.extend(["--method", method])
    checked_run(command, cwd=PROJECT_ROOT)
    checked_run(
        [sys.executable, "-m", "sentiment_manifold", "plot", "--run-dir", FULL_RUN_DIR],
        cwd=PROJECT_ROOT,
    )
else:
    print("Full run is guarded. Set RUN_FULL_REPRODUCTION = True after the smoke run passes.")


## 8. Inspect tables and generated figures

The cell prefers a completed full run and otherwise shows the smoke run. `best_layers.csv` independently selects the maximizing layer for each method × dataset × paper metric, matching this repository's paper-parity reporting rule. Use `patching_records.csv` for the per-pair causal audit trail and `direction_metadata.csv` to locate the fitted direction artifacts.


In [ ]:
import pandas as pd
from IPython.display import Image, display

if (FULL_RUN_DIR / "metrics.csv").exists() and (FULL_RUN_DIR / "best_layers.csv").exists():
    ACTIVE_RUN_DIR = FULL_RUN_DIR
elif (SMOKE_RUN_DIR / "metrics.csv").exists() and (SMOKE_RUN_DIR / "best_layers.csv").exists():
    ACTIVE_RUN_DIR = SMOKE_RUN_DIR
else:
    raise FileNotFoundError("No completed smoke or full run was found.")

print("Displaying:", ACTIVE_RUN_DIR)
metrics = pd.read_csv(ACTIVE_RUN_DIR / "metrics.csv")
best_layers = pd.read_csv(ACTIVE_RUN_DIR / "best_layers.csv")
directions = pd.read_csv(ACTIVE_RUN_DIR / "direction_metadata.csv")
patching = pd.read_csv(ACTIVE_RUN_DIR / "patching_records.csv")

display(best_layers)
display(metrics.head())
display(directions[[column for column in ["method", "layer", "subspace_dimension", "artifact_path"] if column in directions.columns]].head(20))
display(
    patching.groupby(["dataset", "method", "layer"], as_index=False)
    .size()
    .rename(columns={"size": "patched_pairs"})
    .head(30)
)

for figure_name in [
    "table1_best_results.png",
    "toy_logit_diff_percent_by_layer.png",
    "sst_logit_diff_percent_by_layer.png",
]:
    figure_path = ACTIVE_RUN_DIR / "figures" / figure_name
    if figure_path.exists():
        display(Image(filename=str(figure_path)))


## 9. Verify expected artifacts and download the results

This assertion catches incomplete output before packaging. Direction checkpoints are deliberately excluded from the ZIP because they may be large and can already live on Drive.


In [ ]:
expected_files = [
    "resolved_config.json",
    "prompt_manifest.csv",
    "toy_vocabulary.csv",
    "answer_tokens.csv",
    "sst_candidate_manifest.csv",
    "pair_manifest.csv",
    "direction_metadata.csv",
    "patching_records.csv",
    "metrics.csv",
    "best_layers.csv",
]
missing = [name for name in expected_files if not (ACTIVE_RUN_DIR / name).exists()]
assert not missing, f"Incomplete run; missing: {missing}"
print("Verified expected result artifacts.")

import shutil
archive_base = Path("/content") / f"{ACTIVE_RUN_DIR.parent.name}-gpt2-small-results"
archive_path = Path(shutil.make_archive(str(archive_base), "zip", ACTIVE_RUN_DIR))
print("Created:", archive_path)

try:
    from google.colab import files
    files.download(str(archive_path))
except ImportError:
    print("Download helper is available only in Colab.")


## Interpretation boundary

- The full reproduction fits directions on tokenizer-filtered ToyMovieReview **training** examples and evaluates held-out ToyMovieReview plus Pythia-filtered SST.
- `reproduce` follows the paper-parity layer sweep and reports each final metric's best layer independently. If you want leakage-safe layer/hyperparameter selection for a new claim, use the repository's separate `tune` then `confirm` workflow instead of selecting from these final-test results.
- Recovery is intentionally unclipped; values above 100% can occur.
- Random directions are causal controls. Projection or probe performance alone is not evidence that the model uses a direction.
- The optional OpenWebText resample-ablation diagnostic is disabled here and should not be described as the paper's principal OpenWebText result.
